In [1]:
import json
from datetime import datetime, timezone

import pandas as pd
import sempy.fabric as fabric
from sempy.fabric.exceptions import FabricHTTPException

# Display all columns when printing dataframes (helpful for wide tenant settings output)
pd.set_option("display.max_columns", None)

# Constants: keep URLs/endpoints in one place for maintainability
TENANT_SETTINGS_ENDPOINT = "v1/admin/tenantsettings"  # Fabric Admin API path [3](https://learn.microsoft.com/en-us/rest/api/fabric/admin/tenants/list-tenant-settings)
TENANT_SETTINGS_INDEX_URL = "https://learn.microsoft.com/en-us/fabric/admin/tenant-settings-index"  # Settings descriptions table [2](https://learn.microsoft.com/en-us/fabric/admin/tenant-settings-index)


def _fetch_tenant_settings_json(client: fabric.PowerBIRestClient) -> dict:
    """
    Calls the Fabric Admin API to retrieve tenant settings as JSON.

    Notes:
    - Requires Fabric administrator privileges (or equivalent permissions).
    - Raises FabricHTTPException for non-200 responses.
    """
    response = client.get(TENANT_SETTINGS_ENDPOINT)  # GET /v1/admin/tenantsettings [3](https://learn.microsoft.com/en-us/rest/api/fabric/admin/tenants/list-tenant-settings)

    # Sempy doesn't automatically throw on non-200; do it explicitly
    if response.status_code != 200:
        raise FabricHTTPException(response)

    return json.loads(response.text)


def _load_settings_descriptions() -> pd.DataFrame:
    """
    Loads the tenant settings index table from Microsoft Learn into a DataFrame.

    The Learn page contains an HTML table with columns including:
    - "Setting name"
    - "Description"
    """
    # read_html returns a list of dataframes (one per table). The first table is the index.
    tables = pd.read_html(TENANT_SETTINGS_INDEX_URL)  # [2](https://learn.microsoft.com/en-us/fabric/admin/tenant-settings-index)
    descriptions_df = pd.concat(tables, ignore_index=True)

    # Normalize column names expected downstream (defensive coding)
    # Keep original column names if they already match.
    return descriptions_df


def get_fabric_tenant_settings_with_descriptions() -> pd.DataFrame:
    """
    Retrieves Fabric tenant settings and enriches them with human-readable descriptions.

    Output columns:
    - tenantSettingGroup
    - title
    - settingName
    - Description
    - enabled
    - canSpecifySecurityGroups
    - enabledSecurityGroups
    - Settings As Of

    Returns:
    - pd.DataFrame with current tenant settings joined to Learn descriptions.

    Exception behavior:
    - If caller is not a tenant admin / lacks permission, prints a helpful message
      and returns an error string (matching your original behavior).
    """
    rest_client = fabric.PowerBIRestClient()

    try:
        # 1) Fetch settings from the Admin API
        settings_payload = _fetch_tenant_settings_json(rest_client)

        # The blog sample uses `tenantSettings`; the REST doc sample uses `value`.
        # Handle both to be resilient across payload shapes. [1](https://fabric.guru/function-to-get-all-the-fabric-tenant-settings-with-descriptions)[3](https://learn.microsoft.com/en-us/rest/api/fabric/admin/tenants/list-tenant-settings)
        settings_list = (
            settings_payload.get("tenantSettings")
            if "tenantSettings" in settings_payload
            else settings_payload.get("value", [])
        )

        # 2) Flatten the settings JSON into a dataframe
        tenant_settings_df = pd.json_normalize(settings_list)

        # 3) Load descriptions from Learn tenant settings index
        descriptions_df = _load_settings_descriptions()

        # 4) Join settings with descriptions:
        #    - Your original join used settings `title` <-> descriptions `Setting name`.
        #    - We'll keep that default to preserve behavior, but also support joining on `settingName`
        #      if you want to switch later.
        merged_df = (
            tenant_settings_df
            .set_index("title")
            .join(descriptions_df.set_index("Setting name"), how="left")
            .reset_index()
        )

        # 5) Select and order columns consistently
        output_columns = [
            "tenantSettingGroup",
            "title",
            "settingName",
            "Description",
            "enabled",
            "canSpecifySecurityGroups",
            "enabledSecurityGroups",
        ]
        result_df = merged_df[output_columns].copy()

        # 6) Add an "as of" timestamp (UTC-safe; adjust if you prefer local time)
        result_df["Settings As Of"] = datetime.now(timezone.utc).strftime("%d/%m/%Y %H:%M UTC")

        return result_df

    except FabricHTTPException as exc:
        # Keeping your friendly message + returning the exception text
        print("Only tenant admins can use this function\n")
        return str(exc)


# Example usage
tenant_settings_report_df = get_fabric_tenant_settings_with_descriptions()
tenant_settings_report_df

StatementMeta(, 9b5ec002-c728-4df8-a976-74deb098c3d0, 3, Finished, Available, Finished, True)

,tenantSettingGroup,title,settingName,Description,enabled,canSpecifySecurityGroups,enabledSecurityGroups,Settings As Of
0,Admin API settings,Service principals can access read-only admin ...,AllowServicePrincipalsUseReadAdminAPIs,Web apps registered in Microsoft Entra ID can ...,True,True,[{'graphId': 'd3e3c823-98a2-4f0e-8cdd-11c1c12c...,22/05/2026 14:30 UTC
1,Admin API settings,Service principals can access admin APIs used ...,AllowServicePrincipalsUseWriteAdminAPIs,Web apps registered in Microsoft Entra ID can ...,True,True,[{'graphId': 'dca135d1-89ba-41cf-830e-3bb01ccc...,22/05/2026 14:30 UTC
2,Admin API settings,Enhance admin APIs responses with detailed met...,AdminApisIncludeDetailedMetadata,Users and service principals allowed to call P...,True,True,NaN,22/05/2026 14:30 UTC
3,Admin API settings,Enhance admin APIs responses with DAX and mash...,AdminApisIncludeExpressions,Users and service principals eligible to call ...,True,True,NaN,22/05/2026 14:30 UTC
4,Advanced networking,Tenant-level Private Link,AllowAccessOverPrivateLinks,Increase security by allowing people to use a ...,False,False,NaN,22/05/2026 14:30 UTC
...,...,...,...,...,...,...,...,...
160,Workspace settings,Use semantic models across workspaces,UseDatasetsAcrossWorkspaces,Users in the organization can use semantic mod...,True,True,NaN,22/05/2026 14:30 UTC
161,Workspace settings,Block users from reassigning personal workspac...,RestrictMyFolderCapacity,Turn on this setting to prevent users from rea...,False,False,NaN,22/05/2026 14:30 UTC
162,Workspace settings,Define workspace retention period,ConfigureFolderRetentionPeriod,Turn on this setting to define a retention per...,True,False,NaN,22/05/2026 14:30 UTC
163,Workspace settings,Automatically convert and store reports using ...,AutomaticallyUsePBIR,Enable this setting to automatically convert r...,True,True,NaN,22/05/2026 14:30 UTC
